In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

In [2]:
df=pd.read_csv("C:/Users/moham/Downloads/Student_Performance_Dataset (1).csv")

In [3]:
df.head()

,Student_ID,Age,Gender,Class,Study_Hours_Per_Day,Attendance_Percentage,Parental_Education,Internet_Access,Extracurricular_Activities,Math_Score,Science_Score,English_Score,Previous_Year_Score,Final_Percentage,Performance_Level,Pass_Fail
0,S0001,15,Male,12,1.0,65,Postgraduate,No,No,40,39,72,81,50.33,Average,Pass
1,S0002,19,Female,9,1.6,58,Graduate,No,Yes,80,44,35,47,53.00,Average,Pass
2,S0003,14,Female,12,3.6,64,High School,Yes,Yes,83,73,59,58,71.67,Good,Pass
3,S0004,18,Female,9,5.5,68,Postgraduate,Yes,No,68,48,77,54,64.33,Average,Pass
4,S0005,14,Male,10,5.0,80,High School,Yes,No,41,46,36,68,41.00,Poor,Fail


In [4]:
df.shape

(5000, 16)

In [5]:
df.nunique()

Student_ID                    5000
Age                              6
Gender                           2
Class                            4
Study_Hours_Per_Day             56
Attendance_Percentage           51
Parental_Education               3
Internet_Access                  2
Extracurricular_Activities       2
Math_Score                      66
Science_Score                   66
English_Score                   66
Previous_Year_Score             56
Final_Percentage               181
Performance_Level                4
Pass_Fail                        2
dtype: int64

In [6]:
df.isnull().sum()

Student_ID                    0
Age                           0
Gender                        0
Class                         0
Study_Hours_Per_Day           0
Attendance_Percentage         0
Parental_Education            0
Internet_Access               0
Extracurricular_Activities    0
Math_Score                    0
Science_Score                 0
English_Score                 0
Previous_Year_Score           0
Final_Percentage              0
Performance_Level             0
Pass_Fail                     0
dtype: int64

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- 1. Scan all numeric columns ----------
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    return (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)

num_cols = df.select_dtypes(include='number').columns

summary = pd.DataFrame({
    'outliers': df[num_cols].apply(iqr_outliers).sum(),
    'percent': (df[num_cols].apply(iqr_outliers).mean() * 100).round(2),
    'skew': df[num_cols].skew().round(2)
}).sort_values('outliers', ascending=False)
print(summary)

                       outliers  percent  skew
Age                           0      0.0 -0.02
Class                         0      0.0  0.01
Study_Hours_Per_Day           0      0.0 -0.02
Attendance_Percentage         0      0.0  0.00
Math_Score                    0      0.0 -0.03
Science_Score                 0      0.0  0.04
English_Score                 0      0.0 -0.02
Previous_Year_Score           0      0.0  0.01
Final_Percentage              0      0.0 -0.02


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student_ID                  5000 non-null   str    
 1   Age                         5000 non-null   int64  
 2   Gender                      5000 non-null   str    
 3   Class                       5000 non-null   int64  
 4   Study_Hours_Per_Day         5000 non-null   float64
 5   Attendance_Percentage       5000 non-null   int64  
 6   Parental_Education          5000 non-null   str    
 7   Internet_Access             5000 non-null   str    
 8   Extracurricular_Activities  5000 non-null   str    
 9   Math_Score                  5000 non-null   int64  
 10  Science_Score               5000 non-null   int64  
 11  English_Score               5000 non-null   int64  
 12  Previous_Year_Score         5000 non-null   int64  
 13  Final_Percentage            5000 non-null   

In [9]:
import pandas as pd
from scipy.stats import chi2_contingency
target = 'Pass_Fail'
# Select categorical columns
categorical_columns = df.select_dtypes(
    include=['object', 'category', 'bool']
).columns
# Remove the target column
categorical_columns = categorical_columns.drop(
    target, errors='ignore'
)
chi_results = []
for col in categorical_columns:
    # Create a contingency table
    contingency_table = pd.crosstab(df[col], df[target])
    # Perform Chi-Square test
    chi2, p_value, dof, expected = chi2_contingency(
        contingency_table
    )
    chi_results.append({
        'Feature': col,
        'Chi2_Statistic': chi2,
        'P_Value': p_value,
        'Relationship': 'Significant'
        if p_value < 0.05 else 'Not Significant'
    })
# Display results
chi_df = pd.DataFrame(chi_results)
display(chi_df.sort_values(by='P_Value'))

C:\Users\moham\AppData\Local\Temp\ipykernel_7144\684301653.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


,Feature,Chi2_Statistic,P_Value,Relationship
5,Performance_Level,5000.000000,0.000000,Significant
0,Student_ID,5000.000000,0.493351,Not Significant
4,Extracurricular_Activities,0.399617,0.527287,Not Significant
2,Parental_Education,0.645978,0.723982,Not Significant
3,Internet_Access,0.034337,0.852991,Not Significant
1,Gender,0.010437,0.918630,Not Significant


In [10]:
# Identify insignificant categorical features
insignificant_cols = chi_df.loc[
    chi_df['P_Value'] >= 0.05, 'Feature'
].tolist()
# Drop insignificant columns
df = df.drop(columns=insignificant_cols)
print("Dropped columns:", insignificant_cols)
print("Remaining columns:", df.columns.tolist())

Dropped columns: ['Student_ID', 'Gender', 'Parental_Education', 'Internet_Access', 'Extracurricular_Activities']
Remaining columns: ['Age', 'Class', 'Study_Hours_Per_Day', 'Attendance_Percentage', 'Math_Score', 'Science_Score', 'English_Score', 'Previous_Year_Score', 'Final_Percentage', 'Performance_Level', 'Pass_Fail']


In [11]:
import pandas as pd
from scipy.stats import f_oneway
target = 'Pass_Fail'
# Select numerical columns
numerical_columns = df.select_dtypes(
    include='number'
).columns.drop(target, errors='ignore')
anova_results = []
for col in numerical_columns:
    # Select feature values for each class
    group_0 = df.loc[
        df[target].astype(str).str.strip().str.lower() == 'pass',
        col
    ].dropna()
    group_1 = df.loc[
        df[target].astype(str).str.strip().str.lower() == 'fail',
        col
    ].dropna()
    # Check whether both groups contain enough values
    if len(group_0) < 2 or len(group_1) < 2:
        continue
    # Perform ANOVA
    f_stat, p_value = f_oneway(group_0, group_1)
    anova_results.append({
        'Feature': col,
        'F_Statistic': f_stat,
        'P_Value': p_value,
        'Relationship': (
            'Significant' if p_value < 0.05
            else 'Not Significant'
        )
    })

anova_df = pd.DataFrame(anova_results)
print(anova_df.sort_values('P_Value'))

                 Feature  F_Statistic        P_Value     Relationship
8       Final_Percentage  1420.482910  7.906894e-274      Significant
6          English_Score   420.124544   1.023178e-89      Significant
4             Math_Score   402.575398   3.462851e-86      Significant
5          Science_Score   366.289900   7.497760e-79      Significant
7    Previous_Year_Score     2.939837   8.648144e-02  Not Significant
2    Study_Hours_Per_Day     0.819706   3.653099e-01  Not Significant
3  Attendance_Percentage     0.581390   4.458041e-01  Not Significant
1                  Class     0.180703   6.707885e-01  Not Significant
0                    Age     0.139454   7.088399e-01  Not Significant


In [12]:
# Identify insignificant numerical features
insignificant_cols = anova_df.loc[
    anova_df['P_Value'] >= 0.05, 'Feature'
].tolist()
# Drop insignificant columns from the dataset
df = df.drop(columns=insignificant_cols)
print("Dropped columns:", insignificant_cols)
print("Remaining columns:", df.columns.tolist())

Dropped columns: ['Age', 'Class', 'Study_Hours_Per_Day', 'Attendance_Percentage', 'Previous_Year_Score']
Remaining columns: ['Math_Score', 'Science_Score', 'English_Score', 'Final_Percentage', 'Performance_Level', 'Pass_Fail']


In [13]:
df.head()

,Math_Score,Science_Score,English_Score,Final_Percentage,Performance_Level,Pass_Fail
0,40,39,72,50.33,Average,Pass
1,80,44,35,53.00,Average,Pass
2,83,73,59,71.67,Good,Pass
3,68,48,77,64.33,Average,Pass
4,41,46,36,41.00,Poor,Fail


In [14]:
from sklearn.preprocessing import LabelEncoder
Label = LabelEncoder()
df["Performance_Level"]=Label.fit_transform(df["Performance_Level"])
df["Pass_Fail"]=Label.fit_transform(df["Pass_Fail"])

In [15]:
df.head()

,Math_Score,Science_Score,English_Score,Final_Percentage,Performance_Level,Pass_Fail
0,40,39,72,50.33,0,1
1,80,44,35,53.00,0,1
2,83,73,59,71.67,2,1
3,68,48,77,64.33,0,1
4,41,46,36,41.00,3,0


In [16]:
x=df.iloc[:,0:5]
x

,Math_Score,Science_Score,English_Score,Final_Percentage,Performance_Level
0,40,39,72,50.33,0
1,80,44,35,53.00,0
2,83,73,59,71.67,2
3,68,48,77,64.33,0
4,41,46,36,41.00,3
...,...,...,...,...,...
4995,64,60,70,64.67,0
4996,44,59,43,48.67,3
4997,95,41,82,72.67,2
4998,66,45,61,57.33,0


In [17]:
y=df.iloc[:,5]
y

0       1
1       1
2       1
3       1
4       0
       ..
4995    1
4996    0
4997    1
4998    1
4999    1
Name: Pass_Fail, Length: 5000, dtype: int64

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [19]:
x_train

,Math_Score,Science_Score,English_Score,Final_Percentage,Performance_Level
4227,47,95,95,79.00,2
4676,75,84,94,84.33,1
800,91,60,72,74.33,2
3671,41,67,91,66.33,2
4193,35,40,93,56.00,0
...,...,...,...,...,...
4426,50,94,44,62.67,0
466,92,80,88,86.67,1
3092,77,93,44,71.33,2
3772,54,78,55,62.33,0


In [20]:
x_test

,Math_Score,Science_Score,English_Score,Final_Percentage,Performance_Level
1501,76,48,54,59.33,0
2586,36,63,91,63.33,0
2653,43,78,62,61.00,0
1055,94,63,50,69.00,2
705,79,51,88,72.67,2
...,...,...,...,...,...
4711,68,73,96,79.00,2
2313,54,77,91,74.00,2
3214,47,92,40,59.67,0
2732,56,72,69,65.67,0


In [21]:
y_train.value_counts()

Pass_Fail
1    3795
0     205
Name: count, dtype: int64

In [22]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(sampling_strategy='minority')
x_sm, y_sm = smote.fit_resample(x_train, y_train)
y_sm.value_counts()

Pass_Fail
1    3795
0    3795
Name: count, dtype: int64

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
X = df.drop('Pass_Fail', axis=1)
y = df['Pass_Fail']
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
num_cols = X_train.select_dtypes(
    include=['int64', 'float64']
).columns
scaler = MinMaxScaler()
x_train[num_cols] = scaler.fit_transform(X_train[num_cols])
x_test[num_cols] = scaler.transform(X_test[num_cols])

In [24]:
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [25]:
#import the model
#training (x_train,y_train)
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()

In [26]:
model.fit(x_train_scaled,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [27]:
x_pred=model.predict(x_train_scaled)

In [28]:
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(x_pred,y_train)
print("training accuracy is ",train_acc)

training accuracy is  1.0


In [29]:
y_pred=model.predict(x_test_scaled)

In [30]:
from sklearn.metrics import accuracy_score
test_acc = accuracy_score(y_test,y_pred)
print("testing accuracy is ",test_acc)

testing accuracy is  1.0


In [31]:
from sklearn.metrics import classification_report
# Display classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        60
           1       1.00      1.00      1.00       940

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [32]:
from sklearn.neighbors import KNeighborsClassifier
#from sklearn.neighbors import KNeighborsRegressor
model=KNeighborsClassifier()

In [33]:
model.fit(x_train_scaled,y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [34]:
x_pred=model.predict(x_train_scaled)

In [35]:
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(x_pred,y_train)
print("training accuracy is ",train_acc)

training accuracy is  1.0


In [36]:
y_pred=model.predict(x_test_scaled)

In [37]:
from sklearn.metrics import accuracy_score
test_acc = accuracy_score(y_test,y_pred)
print("testing accuracy is ",test_acc)

testing accuracy is  1.0


In [38]:
from sklearn.model_selection import GridSearchCV
KNN_PARAMETER = {'n_neighbors': [3, 5, 7, 11],'metric': ['euclidean', 'manhattan']}
grid_search = GridSearchCV(KNeighborsClassifier(), KNN_PARAMETER, cv=5)
grid_search.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",KNeighborsClassifier()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'metric': ['euclidean', 'manhattan'], 'n_neighbors': [3, 5, ...]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information 

In [39]:
from sklearn.metrics import classification_report
# Display classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        60
           1       1.00      1.00      1.00       940

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [40]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
model = SVC()

In [41]:
model.fit(x_train_scaled,y_train)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [42]:
x_pred=model.predict(x_train_scaled)

In [43]:
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(x_pred,y_train)
print("training accuracy is ",train_acc)

training accuracy is  1.0


In [44]:
y_pred=model.predict(x_test_scaled)

In [45]:
from sklearn.metrics import accuracy_score
test_acc = accuracy_score(y_test,y_pred)
print("testing accuracy is ",test_acc)

testing accuracy is  1.0


In [46]:
from sklearn.metrics import classification_report
# Display classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00        60
           1       1.00      1.00      1.00       940

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [47]:
from sklearn.tree import DecisionTreeClassifier
model=DecisionTreeClassifier(criterion= 'gini', max_depth= None, min_samples_leaf= 1, min_samples_split= 2)

In [48]:
model.fit(x_train_scaled,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [49]:
x_pred=model.predict(x_train_scaled)

In [50]:
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(x_pred,y_train)
print("training accuracy is ",train_acc)

training accuracy is  1.0


In [51]:
y_pred=model.predict(x_test_scaled)

In [52]:
from sklearn.metrics import accuracy_score
test_acc = accuracy_score(y_test,y_pred)
print("testing accuracy is ",test_acc)

testing accuracy is  1.0


In [53]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
param_grid = {'criterion': ['gini', 'entropy'],'max_depth': [None, 3, 5, 10],'min_samples_split': [2, 5, 10],'min_samples_leaf': [1, 2, 4]
}
dt=DecisionTreeClassifier()
# GridSearchCV
grid_search = GridSearchCV(estimator=dt,param_grid=param_grid,cv=5,scoring='accuracy',)
# Train
grid_search.fit(X_train, y_train)
# Best parameters
print("Best Parameters:", grid_search.best_params_)
print("Best CV Accuracy:", grid_search.best_score_)

Best Parameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best CV Accuracy: 1.0


In [54]:
from sklearn.ensemble import RandomForestClassifier
model=RandomForestClassifier(max_depth= 7, min_samples_leaf= 4, min_samples_split= 16, n_estimators= 156)

In [55]:
model.fit(x_train_scaled,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",156
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",7
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",16
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [56]:
x_pred=model.predict(x_train_scaled)

In [57]:
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(x_pred,y_train)
print("training accuracy is ",train_acc)

training accuracy is  1.0


In [58]:
y_pred=model.predict(x_test_scaled)

In [59]:
from sklearn.metrics import accuracy_score
test_acc = accuracy_score(y_test,y_pred)
print("testing accuracy is ",test_acc)

testing accuracy is  1.0


In [60]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
# PARAMETERS
param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(1, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}
# RANDOM SEARCH
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)
# FIT
random_search.fit(x_train, y_train)
# OUTPUT
print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

Best Parameters: {'max_depth': 7, 'min_samples_leaf': 4, 'min_samples_split': 16, 'n_estimators': 156}
Best Score: 1.0
